In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from sklearn.preprocessing import MinMaxScaler
from statsmodels.tsa.arima.model import ARIMA
import warnings


In [ ]:
# Tải Dữ Liệu Từ CSV và Cấu hình dự báo tuần

# 1. Truy vấn dữ liệu thực tế từ tệp data_sales.csv cục bộ
df = pd.read_csv('data_sales.csv')
df['sale_date'] = pd.to_datetime(df['sale_date'])

# 2. Gom nhóm theo tuần (W) để đồng bộ với ứng dụng
weekly_data = df.resample('W', on='sale_date')['revenue'].sum().reset_index()
weekly_data.columns = ['sale_date', 'revenue']
weekly_data = weekly_data.sort_values('sale_date')

# 3. Hàm dùng chung để lưu kết quả dự báo vào tệp forecasts.csv cục bộ
def save_forecast_to_csv(forecast_dates, predictions, model_name):
    import os
    now_str = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    new_rows = []
    for i, pred in enumerate(predictions):
        forecast_date_str = forecast_dates[i].strftime('%Y-%m-%d')
        predicted_val = max(0.0, float(pred))
        new_rows.append({
            'product_id': 1,
            'forecast_date': forecast_date_str,
            'predicted_revenue': predicted_val,
            'model_name': model_name,
            'created_at': now_str
        })
    df_new = pd.DataFrame(new_rows)
    if os.path.exists('forecasts.csv'):
        df_fc = pd.read_csv('forecasts.csv')
        # Loại bỏ các dự báo cũ của mô hình này để tránh trùng lặp
        df_fc = df_fc[df_fc['model_name'] != model_name]
        df_fc = pd.concat([df_fc, df_new], ignore_index=True)
    else:
        df_fc = df_new
    df_fc.to_csv('forecasts.csv', index=False, encoding='utf-8')


In [ ]:
# Huấn luyện và dự báo với Linear Regression, Random Forest & XGBoost (Dùng Đặc Trưng Trễ)
import numpy as np
import pandas as pd
import warnings
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from datetime import datetime, timedelta

warnings.filterwarnings('ignore')

# 1. Tạo các đặc trưng trễ động từ dữ liệu tuần
df_feat = weekly_data.copy()
df_feat['lag_1'] = df_feat['revenue'].shift(1)
df_feat['lag_2'] = df_feat['revenue'].shift(2)
df_feat['lag_4'] = df_feat['revenue'].shift(4)
df_feat['rolling_mean_4'] = df_feat['revenue'].rolling(window=4).mean()
df_feat['month_num'] = df_feat['sale_date'].dt.month
df_feat['week_of_year'] = df_feat['sale_date'].dt.isocalendar().week.astype(int)

# Bỏ qua NaN ở các dòng đầu
df_feat_clean = df_feat.dropna().reset_index(drop=True)
feature_cols = ['lag_1', 'lag_2', 'lag_4', 'rolling_mean_4', 'month_num', 'week_of_year']

y = df_feat_clean['revenue'].values
X = df_feat_clean[feature_cols]
dates_all = df_feat_clean['sale_date'].values

N = len(y)
test_len = 42
train_len = N - test_len

# Chia tập Train/Test theo đặc trưng trễ
y_train, y_test = y[:train_len], y[train_len:]
X_train, X_test = X.iloc[:train_len], X.iloc[train_len:]

# Đánh giá Linear Regression (Dự báo 1 bước)
lr_model_eval = LinearRegression().fit(X_train, y_train)
lr_pred_eval = lr_model_eval.predict(X_test)
lr_mae = mean_absolute_error(y_test, lr_pred_eval)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_pred_eval))
print(f"Linear Regression Evaluation - MAE: {lr_mae:.2f}, RMSE: {lr_rmse:.2f}")

# Đánh giá Random Forest (Dự báo 1 bước)
rf_model_eval = RandomForestRegressor(n_estimators=100, random_state=42).fit(X_train, y_train)
rf_pred_eval = rf_model_eval.predict(X_test)
rf_mae = mean_absolute_error(y_test, rf_pred_eval)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred_eval))
print(f"Random Forest Evaluation - MAE: {rf_mae:.2f}, RMSE: {rf_rmse:.2f}")

# Đánh giá XGBoost (Dự báo 1 bước)
xgb_model_eval = XGBRegressor(n_estimators=100, learning_rate=0.05, max_depth=5, random_state=42).fit(X_train, y_train)
xgb_pred_eval = xgb_model_eval.predict(X_test)
xgb_mae = mean_absolute_error(y_test, xgb_pred_eval)
xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_pred_eval))
print(f"XGBoost Evaluation - MAE: {xgb_mae:.2f}, RMSE: {xgb_rmse:.2f}")

# Hàm phụ trợ dự báo đệ quy tương lai 12 tuần
def forecast_recursive(model, df_feat, num_steps=12):
    history = list(df_feat['revenue'].values)
    last_date = df_feat['sale_date'].max()
    predictions = []
    for i in range(num_steps):
        next_date = last_date + pd.Timedelta(weeks=i+1)
        month_num = next_date.month
        week_of_year = int(next_date.isocalendar()[1])
        l1 = history[-1]
        l2 = history[-2]
        l4 = history[-4]
        rm4 = np.mean(history[-4:])
        X_next = pd.DataFrame([{
            'lag_1': l1,
            'lag_2': l2,
            'lag_4': l4,
            'rolling_mean_4': rm4,
            'month_num': month_num,
            'week_of_year': week_of_year
        }])
        X_next = X_next[feature_cols]
        pred = max(0.0, float(model.predict(X_next)[0]))
        predictions.append(pred)
        history.append(pred)
    return np.array(predictions)

# Huấn luyện trên 100% dữ liệu để dự báo tương lai
lr_model = LinearRegression().fit(X, y)
rf_model = RandomForestRegressor(n_estimators=100, random_state=42).fit(X, y)
xgb_model = XGBRegressor(n_estimators=100, learning_rate=0.05, max_depth=5, random_state=42).fit(X, y)

# Dự báo đệ quy tương lai 12 tuần
lr_predictions = forecast_recursive(lr_model, df_feat_clean, 12)
rf_predictions = forecast_recursive(rf_model, df_feat_clean, 12)
xgb_predictions = forecast_recursive(xgb_model, df_feat_clean, 12)

# Lưu dự báo tương lai vào CSV cục bộ
last_date = pd.to_datetime(df_feat_clean['sale_date'].max())
future_dates = [last_date + timedelta(weeks=i+1) for i in range(12)]

print("--- Lưu Dự Báo Linear Regression ---")
save_forecast_to_csv(future_dates, lr_predictions, 'Linear Regression')

print("--- Lưu Dự Báo Random Forest ---")
save_forecast_to_csv(future_dates, rf_predictions, 'Random Forest')

print("--- Lưu Dự Báo XGBoost ---")
save_forecast_to_csv(future_dates, xgb_predictions, 'XGBoost')

print("Thành công! Đã cập nhật tệp forecasts.csv với các mô hình sử dụng đặc trưng trễ.")

In [ ]:
# Huấn luyện và dự báo với LSTM (Deep Learning)
# Chuẩn hóa dữ liệu
scaler = MinMaxScaler(feature_range=(0, 1))
y_scaled = scaler.fit_transform(y.reshape(-1, 1)).flatten()

# Đánh giá LSTM trên Test set 42 tuần
y_train_scaled = scaler.fit_transform(y_train.reshape(-1, 1)).flatten()
y_test_scaled = scaler.transform(y_test.reshape(-1, 1)).flatten()

def create_sequences(data, seq_length):
    X_seq, y_seq = [], []
    for i in range(len(data) - seq_length):
        X_seq.append(data[i:i+seq_length])
        y_seq.append(data[i+seq_length])
    return np.array(X_seq), np.array(y_seq)

seq_length = 3
X_train_lstm, y_train_lstm = create_sequences(y_train_scaled, seq_length)

# Tạo input sequence cho Test set
y_test_seq_input = np.concatenate([y_train_scaled[-seq_length:], y_test_scaled])
X_test_lstm, y_test_lstm = create_sequences(y_test_seq_input, seq_length)

# Khởi tạo và fit mô hình evaluation
lstm_eval = Sequential()
lstm_eval.add(LSTM(50, activation='relu', input_shape=(seq_length, 1)))
lstm_eval.add(Dense(1))
lstm_eval.compile(optimizer='adam', loss='mse')
lstm_eval.fit(X_train_lstm.reshape(-1, seq_length, 1), y_train_lstm, epochs=50, verbose=0)

# Predict và đánh giá
lstm_pred_scaled = lstm_eval.predict(X_test_lstm.reshape(-1, seq_length, 1), verbose=0)
lstm_pred = scaler.inverse_transform(lstm_pred_scaled).flatten()
lstm_mae = mean_absolute_error(y_test, lstm_pred)
lstm_rmse = np.sqrt(mean_squared_error(y_test, lstm_pred))
print(f"LSTM Evaluation (Weekly) - MAE: {lstm_mae:.2f}, RMSE: {lstm_rmse:.2f}")

# Retrain LSTM trên toàn bộ dữ liệu (100%)
scaler_full = MinMaxScaler(feature_range=(0, 1))
y_all_scaled = scaler_full.fit_transform(y.reshape(-1, 1)).flatten()
X_all_lstm, y_all_lstm = create_sequences(y_all_scaled, seq_length)

lstm_full = Sequential()
lstm_full.add(LSTM(50, activation='relu', input_shape=(seq_length, 1)))
lstm_full.add(Dense(1))
lstm_full.compile(optimizer='adam', loss='mse')
lstm_full.fit(X_all_lstm.reshape(-1, seq_length, 1), y_all_lstm, epochs=50, verbose=0)

# Dự báo đệ quy 12 tuần tương lai
last_seq = y_all_scaled[-seq_length:].reshape(1, seq_length, 1)
lstm_predictions_scaled = []
for _ in range(12):
    next_scaled = lstm_full.predict(last_seq, verbose=0)
    lstm_predictions_scaled.append(next_scaled[0][0])
    last_seq = np.append(last_seq[:, 1:, :], next_scaled.reshape(1, 1, 1), axis=1)

lstm_predictions = scaler_full.inverse_transform(np.array(lstm_predictions_scaled).reshape(-1, 1)).flatten()

# Lưu dự báo vào tệp CSV cục bộ
print("--- Lưu Dự Báo LSTM ---")
save_forecast_to_csv(future_dates, lstm_predictions, 'LSTM')
print("Thành công! Đã cập nhật tệp forecasts.csv với LSTM.")

In [ ]:
# Huấn luyện và dự báo với ARIMA
# 1. Đánh giá ARIMA trên tập Test (Sử dụng walk-forward một bước)
history = list(y_train)
arima_pred_eval_list = []
for t in range(test_len):
    model = ARIMA(history, order=(1, 1, 1))
    model_fit = model.fit()
    output = model_fit.forecast()
    arima_pred_eval_list.append(output[0])
    history.append(y_test[t])
arima_pred_eval = np.array(arima_pred_eval_list)

arima_mae = mean_absolute_error(y_test, arima_pred_eval)
arima_rmse = np.sqrt(mean_squared_error(y_test, arima_pred_eval))
print(f"ARIMA Evaluation (Weekly) - MAE: {arima_mae:.2f}, RMSE: {arima_rmse:.2f}")

# 2. Retrain ARIMA trên toàn bộ dữ liệu (y) để dự báo 12 tuần tới
arima_model_full = ARIMA(y, order=(1, 1, 1))
arima_model_full_fit = arima_model_full.fit()
arima_predictions = arima_model_full_fit.forecast(steps=12)

# Lưu dự báo vào tệp CSV cục bộ
print("--- Lưu Dự Báo ARIMA ---")
save_forecast_to_csv(future_dates, arima_predictions, 'ARIMA')
print("Thành công! Đã cập nhật tệp forecasts.csv với ARIMA.")

In [ ]:
# Vẽ biểu đồ hiển thị kết quả dự báo đầy đủ 5 mô hình
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(12, 5))

# 1. Vẽ dữ liệu lịch sử thực tế (20 tuần cuối)
hist_dates = weekly_data['sale_date'][-20:].values
hist_revenue = weekly_data['revenue'][-20:].values
plt.plot(hist_dates, hist_revenue, label='Lịch sử (Actual Sales)', marker='o', color='#38BDF8', linewidth=2.5)

# Lấy ngày cho trục hoành dự báo
forecast_dates_plot = [weekly_data['sale_date'].max() + timedelta(weeks=i+1) for i in range(12)]

# Cấu hình màu và marker cho từng mô hình
forecast_data = {
    'Linear Regression': lr_predictions,
    'Random Forest': rf_predictions,
    'XGBoost': xgb_predictions,
    'LSTM': lstm_predictions,
    'ARIMA': arima_predictions
}

colors_map = {
    'Linear Regression': '#F59E0B',
    'Random Forest': '#10B981',
    'XGBoost': '#14B8A6',
    'LSTM': '#EC4899',
    'ARIMA': '#A855F7'
}

markers_map = {
    'Linear Regression': 's',
    'Random Forest': '^',
    'XGBoost': 'p',
    'LSTM': 'o',
    'ARIMA': 'd'
}

# 2. Vẽ các đường dự báo nối tiếp mốc lịch sử cuối cùng
for name, preds in forecast_data.items():
    # Kết nối điểm lịch sử cuối cùng với điểm dự báo đầu tiên
    conn_dates = np.concatenate([[hist_dates[-1]], forecast_dates_plot])
    conn_dates = pd.to_datetime(conn_dates)
    conn_preds = np.concatenate([[hist_revenue[-1]], preds])
    
    plt.plot(
        conn_dates, 
        conn_preds, 
        label=f'Dự báo {name}', 
        linestyle='--', 
        marker=markers_map.get(name, 'o'), 
        color=colors_map.get(name, '#94A3B8'),
        linewidth=1.8
    )

plt.title('Đường dự báo 12 tuần tương lai của cả 5 mô hình (Nối tiếp dữ liệu lịch sử)')
plt.xlabel('Thời gian')
plt.ylabel('Doanh thu ($)')
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()
